In [1]:
import json
import time
from urllib.parse import urljoin
from bs4 import BeautifulSoup
from google import genai
import ipywidgets as widgets
import pandas as pd
import requests
from tqdm.notebook import tqdm
import platform
import urllib.request

client = genai.Client(api_key="AIzaSyDqdEqzQ1CVBkvOM")

In [2]:
base_url_w = widgets.Text(value="https://djinni.co/jobs")
sub_url_w = widgets.Text(
    value="?search_type=basic-search&primary_keyword=Data%20Analyst"
)
count_w = widgets.IntSlider(value=2, min=1, max=20)
sleep_w = widgets.IntSlider(value=6, min=1, max=30)

main_selector_w = widgets.Text(value="main#jobs_main")
link_selector_w = widgets.Text(value="ul div a[href]")
url_pattern_w = widgets.Text(value="/jobs/")
content_selector_w = widgets.Text(value="div.page-content")

prompt = """Extract structured data from this job vacancy text into a valid JSON object.
Do not include markdown or explanations. Return empty string "" if field missing.

Schema:
{
  "title": "", "company_name": "", "company_description": "", "role_type": "",
  "seniority": "", "location": "", "posted_date": "", "job_type": "",
  "salary": "", "salary_prediction": "", "skills_required": [],
  "skills_preferred": [], "tech_stack": [], "keywords": [],
  "requirements": [], "benefits": []
}"""

prompt_w = widgets.Textarea(
    value=prompt, layout=widgets.Layout(height="200px")
)

display(
    widgets.VBox(
        [
            widgets.Label("base url:"),
            base_url_w,
            widgets.Label("search query:"),
            sub_url_w,
            widgets.Label("jobs count:"),
            count_w,
            widgets.Label("sleep time (sec):"),
            sleep_w,
            widgets.Label("main selector (dom):"),
            main_selector_w,
            widgets.Label("link selector (dom):"),
            link_selector_w,
            widgets.Label("url filter pattern:"),
            url_pattern_w,
            widgets.Label("page content selector (dom):"),
            content_selector_w,
            widgets.Label("ai prompt system instructions:"),
            prompt_w,
        ]
    )
)

In [3]:
def get_headers():
    py_ver = platform.python_version()
    os_info = platform.system()
    default = urllib.request.URLopener.version

    return {
        "User-Agent": f"Mozilla/5.0 ({os_info}; OpenBot) Python/{py_ver} {default}"
    }


def fetch_job_links(sub_url):
    base_url = base_url_w.value
    res = requests.get(f"{base_url}/{sub_url}", headers=get_headers())
    res.raise_for_status()
    soup = BeautifulSoup(res.text, "lxml")

    main = soup.select_one(main_selector_w.value)
    if not main:
        return []

    links = [
        urljoin(base_url, a.get("href"))
        for a in main.select(link_selector_w.value)
        if url_pattern_w.value in a.get("href", "")
    ]
    return list(set(links))


def fetch_job_pages_content(url):
    try:
        res = requests.get(url, headers=get_headers(), timeout=15)
        res.raise_for_status()
        soup = BeautifulSoup(res.text, "lxml")
        div = soup.select_one(content_selector_w.value)
        return div.get_text(separator="\n", strip=True) if div else ""
    except Exception as ex:
        print(f"  page error: {ex}")
        return ""


def fetch_ai_response(content):
    prompt = f"{prompt_w.value}\n\nText: {content}"
    res = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=prompt,
        config={"response_mime_type": "application/json"},
    )
    return res

In [4]:
def generate_df():
    sub_url, count, sleep_time = (
        sub_url_w.value,
        count_w.value,
        sleep_w.value,
    )
    print(f"fetching job links for: {sub_url}")
    job_links = fetch_job_links(sub_url)[:count]
    print(f"found {len(job_links)} jobs to process")

    ai_responses = []

    for job_link in tqdm(job_links, desc="processing jobs"):
        print(f"\nprocessing: {job_link}")
        while True:
            try:
                print("fetching page content")
                content = fetch_job_pages_content(job_link)
                print("sending to ai")
                ai_res = fetch_ai_response(content)
                ai_responses.append(ai_res.text)
                print(f"done: {job_link}")
                break
            except Exception as ex:
                print(f"error: {ex}")
                print(f"retrying in {sleep_time} seconds")
                time.sleep(sleep_time)
        time.sleep(sleep_time)

    print("\nbuilding dataframe")
    parsed = [json.loads(item) for item in ai_responses]
    df = pd.json_normalize(parsed)
    df["links"] = job_links
    return df

In [6]:
df = generate_df()
display(df.head())

fetching job links for: ?search_type=basic-search&primary_keyword=Data%20Analyst
found 2 jobs to process


processing jobs:   0%|          | 0/2 [00:00<?, ?it/s]


processing: https://djinni.co/jobs/819463-bi-analyst/
fetching page content
sending to ai
done: https://djinni.co/jobs/819463-bi-analyst/

processing: https://djinni.co/jobs/826110-product-data-analyst/
fetching page content
sending to ai
done: https://djinni.co/jobs/826110-product-data-analyst/

building dataframe


,title,company_name,company_description,role_type,seniority,location,posted_date,job_type,salary,salary_prediction,skills_required,skills_preferred,tech_stack,keywords,requirements,benefits,links
0,Senior BI Analyst,Add Talent Solutions,We’re building a next-generation iGaming platf...,Product,Senior,Full Remote Worldwide,17 April,Fulltime,$3000-5000,,"[Tableau expertise, SQL, data modeling skills,...",[iGaming / betting / high-scale transactional ...,"[Tableau, SQL, Python, Snowflake, BigQuery, Re...","[Data Analyst, BI Analyst, iGaming, Data pipel...",[5+ years in BI / Data / Analytics Engineering...,"[Real ownership, Work on AI + modern data stac...",https://djinni.co/jobs/819463-bi-analyst/
1,Product/Data Analyst,QUESTERA,We are looking for a Product/Data Analyst with...,Product/Data Analyst,Mid-level,Full Remote,19 May,Fulltime,$1500-3000,,[Strong SQL skills for querying and analyzing ...,"[analytics, preferably (but not necessarily) i...","[MS SQL, MongoDB, Google Analytics, Power BI, ...","[Product, Data, Analytics, Gamedev, Ukrainian,...",[2+ years of experience in data analytics or p...,[Work on a cutting-edge Gaming platform tailor...,https://djinni.co/jobs/826110-product-data-ana...
